# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [1]:
%pip install -Uqqq langchain_openai langchain_community langchain_tavily langgraph wikipedia numexpr arxiv ddgs

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

In [3]:
import importlib, pkgutil

package = importlib.import_module('langchain_community.tools')

for module in pkgutil.iter_modules(package.__path__):
    print(module.name)

ainetwork
amadeus
arxiv
asknews
audio
azure_ai_services
azure_cognitive_services
bearly
bing_search
brave_search
cassandra_database
clickup
cogniswitch
connery
convert_to_openai
databricks
dataforseo_api_search
dataherald
ddg_search
e2b_data_analysis
edenai
eleven_labs
few_shot
file_management
financial_datasets
github
gitlab
gmail
golden_query
google_books
google_cloud
google_finance
google_jobs
google_lens
google_scholar
google_serper
google_trends
graphql
human
ifttt
interaction
jina_search
jira
json
memorize
merriam_webster
metaphor_search
mojeek_search
multion
nasa
nuclia
office365
openai_dalle_image_generation
openapi
openweathermap
passio_nutrition_ai
playwright
plugin
polygon
powerbi
pubmed
render
requests
riza
scenexplain
searchapi
searx_search
semanticscholar
shell
slack
sleep
spark_sql
sql_database
stackexchange
steam
steamship_image_generation
tavily_search
vectorstore
wikidata
wikipedia
wolfram_alpha
yahoo_finance_news
you
youtube
zapier
zenguard


In [4]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
print(wiki_tool.run('PhysicalAI'))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '걸그룹 튜이드 멤버 알려줘')]

llm = init_chat_model('gpt-5.4-mini')
# print(llm.invoke('걸그룹 튜이드 멤버 알려줘'))  # 최신 정보 모름

agent = create_agent(
    model=llm,
    tools=[wiki_tool]
)

response = agent.invoke({'messages': messages})
pprint(response)

{'messages': [HumanMessage(content='걸그룹 튜이드 멤버 알려줘', additional_kwargs={}, response_metadata={}, id='5ada7e70-8bde-4941-ae2a-f12e986557ed'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 173, 'total_tokens': 193, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHLLGOyZCddK7BzEkRx79Kocz472k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04152-8943-72c3-8ca9-fbcc3a6c0130-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Tweed girl group members'}, 'id': 'call_mTrITWwBtEB0tgweY

In [ ]:
print(response['messages'][-1].content)

제가 확인한 바로는 **“튜이드”라는 이름의 걸그룹**은 일반적으로 알려진 정보가 없어요.  
혹시 **Tweed / Tuid / 투이드**처럼 철자가 비슷한 다른 그룹을 말씀하신 걸까요?

원하시면 제가 바로 찾아드릴 수 있게 아래 중 하나를 알려주세요:
- 그룹의 **영문 이름**
- **멤버 이름 일부**
- **데뷔 시기**
- **나라/소속사**

알려주시면 멤버 목록 정리해서 답해드릴게요.


### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [ ]:
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

llm = init_chat_model('gpt-5.4-mini')
tools = load_tools(['arxiv', 'wikipedia'])

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt='주어진 도구 활용해 사용자 질문에 거짓없이 답변 작성'
)

messages = [('human', '9711003 이 논문 내용 간단히 설명 (한글답변)')]
response = agent.invoke({'messages': messages})
pprint(response)
print('=' * 30)
print(response['messages'][-1].content)

C:\Users\playdata2\AppData\Local\Temp\ipykernel_20316\1062476266.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.agent_toolkits.load_tools import load_tools


AttributeError: object has no attribute 'status'

In [ ]:
!pip list

Package                   Version
------------------------- ---------------
aiohappyeyeballs          2.7.1
aiohttp                   3.14.3
aiosignal                 1.4.0
altair                    6.2.2
annotated-doc             0.0.5
annotated-types           0.8.0
anyio                     4.14.2
arxiv                     1.4.8
asttokens                 3.0.2
attrs                     26.1.0
beautifulsoup4            4.15.0
blinker                   1.9.0
certifi                   2026.7.22
cffi                      2.1.1
charset-normalizer        3.5.1
click                     8.4.2
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
curl_cffi                 0.16.0
cycler                    0.12.1
ddgs                      9.16.0
debugpy                   1.8.21
distro                    1.9.0
dotenv                    0.9.9
executing                 2.2.1
faiss-cpu                 1.15.0
feedparser                6.0.14
feedparser-sgm

In [ ]:
import requests
import xml.etree.ElementTree as ET
from langchain_core.tools import tool

@tool
def search_arxiv(arxiv_id: str) -> str:
    """arXiv 논문 ID로 제목, 저자, 초록 조회"""

    url = "https://export.arxiv.org/api/query"
    response = requests.get(
        url,
        params = {
            'id_list': arxiv_id,
            'max_results': 1
        },
        timeout = 10
    )

    response.raise_for_status()

    root = ET.fromstring(response.text)
    ns = {'atom': "https://www.w3.org/2005/Atom"}
    entry = root.find('atom:entry', ns)

    if entry is None:
        return 'Not Found'

    title = entry.findtext('atom:title', namespaces=ns).strip()
    summary = entry.findtext('atom:summary', namespaces=ns).strip()
    authors = [author.findtext('atom:name', namespaces=ns) for author in entry.findall('atom:author', ns)]

    return f"""
Title: {title}
Authors: {', '.join(authors)}
Abstract: {summary}
"""

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from pprint import pprint

wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

tools = [search_arxiv, wiki_tool]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt='주어진 도구 활용해 사용자 질문에 거짓없이 답변 작성'
)

messages = [('human', '1706.03762 논문 간단히 한글로 설명')]
response = agent.invoke({'messages': messages})
pprint(response['messages'][-1].content)

('arXiv **1706.03762**는 잘 알려진 **“Attention Is All You Need”** 논문입니다.  \n'
 '이 논문은 **트랜스포머(Transformer)** 구조를 제안한 것으로, 이후 번역·요약·언어모델 등 자연어처리의 표준이 되었습니다.\n'
 '\n'
 '간단히 말하면:\n'
 '\n'
 '- 기존의 RNN/LSTM처럼 **순서대로 문장을 처리하지 않고**\n'
 '- **어텐션(attention)**만으로 단어들 사이의 관계를 한 번에 계산합니다.\n'
 '- 그래서 **병렬 처리**가 잘 되고, **학습 속도와 성능**이 좋아졌습니다.\n'
 '- 특히 **Self-Attention**을 통해 문장 안에서 각 단어가 다른 단어들과 얼마나 관련 있는지 파악합니다.\n'
 '\n'
 '핵심 아이디어:\n'
 '1. **입력 문장**을 단어 단위로 벡터화\n'
 '2. 각 단어가 다른 단어들을 참고하도록 **Self-Attention** 적용\n'
 '3. 이런 층을 여러 번 쌓아 문맥을 깊게 이해\n'
 '4. 번역 같은 작업에서는 **Encoder-Decoder** 구조로 사용\n'
 '\n'
 '한 줄 요약:  \n'
 '**“문장을 순차적으로 읽는 대신, 모든 단어의 관계를 동시에 계산해서 더 빠르고 강력한 언어 모델을 만든 논문”**입니다.\n'
 '\n'
 '원하시면 제가 이 논문을 **그림처럼 쉽게**, 또는 **수식 없이 더 쉽게**도 설명해드릴게요.')


In [ ]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-5.6-luna')
tools = load_tools(['wikipedia', 'llm-math'], llm=llm)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt='주어진 도구 활용해 사용자 질문에 공손하게 답변 작성. 수학 계산은 반드시 llm-math 툴 사용'
)

response = agent.invoke({'messages': '3.5의 세제곱 값과 그 결과에 5를 곱한 값'})
pprint(response['messages'][-1].content)

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "Function tools with reasoning_effort are not supported for gpt-5.6-luna in /v1/chat/completions. To use function tools, use /v1/responses or set reasoning_effort to 'none'.", 'type': 'invalid_request_error', 'param': 'reasoning_effort', 'code': None}}

### duckduckgo
https://docs.langchain.com/oss/python/integrations/tools/ddg

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults, DuckDuckGoSearchRun

ddgs = DuckDuckGoSearchRun()
pprint(ddgs.invoke("Trump's first name?"))
print()

ddgs2 = DuckDuckGoSearchResults()
pprint(ddgs2.invoke("Trump's first name?"))

('2026. 7. 17. · The inauguration of Donald Trump as the 45th president of the '
 'United States marked the commencement of the first term of Trump as '
 'president and the only ... 2025. 11. 26. · Donald John Trump (b. June 14, '
 '1946, in Queens, New York) is the 47th and 45th president of the United '
 'States. Trump served his first term from January 20, ... 2025. 10. 14. · '
 "Literal Meaning of Donald Trump's Full Name Donald J. Trump's full name, "
 'Donald John Trump, carries layered meanings: Donald: Scottish Gaelic origin, '
 '... 2026. 6. 24. · A report suggests that the popularity of the name '
 '“Donald” has declined significantly since Donald Trump returned to office. '
 "It noted that the name peaked ... 2026. 8. 6. · Trump's name and face are "
 'showing up everywhere during his second term. Take a look.')

('snippet: The company name "E. Trump & Son" appeared in advertising by 1924, '
 '[44] by which year Trump ostensibly used an $800 loan from his mother to '
 'compl

In [ ]:
llm = init_chat_model('gpt-5.4-mini')
tools = [ddgs2]

agent = create_agent(llm, tools)

response = agent.invoke({'messages': 'gs25 민음사 빵 정보를 툴 이용해서 확인'})

pprint(response)
print('='*30)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='gs25 민음사 빵 정보를 툴 이용해서 확인', additional_kwargs={}, response_metadata={}, id='19bb4848-471c-4ae8-a91b-a8a1e9b767ed'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 169, 'total_tokens': 197, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHNsqtVZP3erokHhV8inlgyaufcgu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041e7-9637-7b20-bc57-26f9654ccdb8-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'GS25 민음사 빵 정보'}, 'id': 'call_3im1

In [ ]:
ddgs2.invoke('GS25 민음사 빵')

'snippet: 4 days ago · GS25 민음사 문학빵, 구하기가 어렵더라고요. 저는 이거 구하려고 진짜 여러 GS25를 돌아다녔어요. 문학빵 성공한 구매 꿀팁 알려드릴게요! 냉장 빵 입고되는 시간을 노리자!, title: GS25 민음사 문학빵 종류 총정리! 책갈피 20종 및 구매 꿀팁 : 네이버..., link: https://blog.naver.com/bebe01_/224387500296, snippet: Aug 20, 2026 · 이번 GS25 민음사 문학빵에서 가장 눈길을 끄는 건 사실 빵 포장지도 예쁘지만 책갈피도 탐이 납니다. 빵 봉지 안에는 투명 책갈피 20종 가운데 1종이 랜덤으로 들어가고, 여기에 GS25 한정 미공개 책갈피 3종까지 포함됩니다., title: GS25 X 민음사 문학빵 1차 8/21 출시! 가격 및 책갈피 1+1 행사 총정..., link: https://blog.naver.com/kimmari02/224384935367, snippet: 6 days ago · 총평!! 달고 맛있다 우유는 필수!!!!!! 책갈피가 너무너므귀엽다 따로 팔아주시길... 빵은 그만 먹고싶어요...ㅋㅋㅋㅠㅠ 1+1이라 좋지만 남은걸 언제 먹을지 고민되네요 여러분도 GS25 신상! GS25 민음사 콜라보 빵 드셔보시는 게 어떠신가요? 추천드릴게요♡, title: [GS25X민음사_민음사빵_신상후기] : 네이버 블로그, link: https://m.blog.naver.com/vip4481/224385769908, snippet: 6 days ago · GS25에서 민음사 책 표지와 똑같이 생긴 민음사빵이 나왔어요! 바로 GS25 X 민음사 X 오늘의 귀여움 콜라보인데요 실제 민음사 문학 작품의 표지를 그대로 살린 패키지라 진짜 똑같더라고요!!! 게다가 빵 하나마다 민음사 투명 책갈피까지 랜덤으로 들어 있습니다!, title: GS25 민음사 빵 사랑에 대하여 모카번 후기 가격 칼로리 책갈피 총정..., link: https://m.blog.n

### tavily-search

https://docs.langchain.com/oss/python/integrations/tools/tavily_search

In [ ]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=3,
    topic='general',
    include_images=True,
    search_depth='advanced'
)

tavily_tool.invoke('2026년 8월 대한민국에서 가장 핫한 이슈')

{'query': '2026년 8월 대한민국에서 가장 핫한 이슈',
 'follow_up_questions': None,
 'answer': None,
 'images': ['https://lookaside.instagram.com/seo/google_widget/crawler/?media_id=3812539243684576662',
  'https://lookaside.fbsbx.com/lookaside/crawler/threads/DcSmY7mkpD4/0/image.jpg',
  'https://cdn.wakeupnews.co.kr/news/photo/202601/973_1856_5154.png',
  'https://lookaside.instagram.com/seo/google_widget/crawler/?media_id=3970609243397805454',
  'https://lookaside.fbsbx.com/lookaside/crawler/threads/DaUU-q_CPuY/0/image.jpg'],
 'results': [{'url': 'https://www.newsshin.co.kr/news/articleView.html?idxno=325554',
   'title': "【뉴스신ㅣ2026년 8월 22일(토) ㅣ대한민국 '핫' 이슈】 < 기자수첩 < 뉴스신 광장 < 기사본문 - 뉴스신(NEWSSHIN)",
   'content': '☞ 뉴스신 기자의 시선  \n 국민은 경기 결과보다 그 안에서 희망을 찾는다.\n\n▣ 금융·가계  \n "빚의 시대, 관리 능력이 생존력"  \n ▶ 브리핑  \n 가계부채와 금리 부담은 여전히 경제의 핵심 변수다.  \n → 구조 분석  \n 금융 안정은 국가 경제 안정과 직결된다.\n\n☞ 뉴스신 기자의 시선  \n 돈을 버는 능력만큼 중요한 시대가 왔다.  \n 돈을 지키는 능력이다.\n\n▣ 미래 대한민국  \n "AI 시대 국가 경쟁력은 사람이다"  \n ▶ 브리핑  \n AI·반도체·첨단산업 경쟁이 국가 

In [ ]:
llm = init_chat_model('gpt-5.4-mini')
tools = [tavily_tool]

agent = create_agent(llm, tools)

response = agent.invoke({'messages': 'AI업계 가장 최신 기술 툴 사용해서 답변'})

pprint(response)
print('='*30)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='AI업계 가장 최신 기술 툴 사용해서 답변', additional_kwargs={}, response_metadata={}, id='4945c68e-cf20-4011-a80b-31b14f324ca5'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 1281, 'total_tokens': 1335, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHO1gZw5ALnGbszu14L3xp0NHN1rI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041ef-ecda-7a83-8e71-e6986005d501-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'latest AI industry tools 2026 newest model

In [ ]:
llm = init_chat_model('gpt-5.4-mini')
tools = [tavily_tool]

agent = create_agent(llm, tools, system_prompt='''
당신은 미국주식시장 분석봇입니다.
사용자가 요청한 기업에 대한 2026년 보고서를 직관적으로 분석해주세요.

# 출력형식
다음 내용을 포함해 표형식 출력 (분석기관별 레코드로 작성)

1. 분석기관명
2. 목표주가범위 (최저 ~ 최대)
3. 전망근거 키워드
4. 신뢰도 지수(1 ~ 10)
''')

response = agent.invoke({'messages': '2096년 상승할 가능성 가장 높은 미국주식 분석'})

pprint(response)
print('='*30)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='2096년 상승할 가능성 가장 높은 미국주식 분석', additional_kwargs={}, response_metadata={}, id='3f7882bd-86c7-4caa-bfdd-992b9d2b5f05'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 1381, 'total_tokens': 1444, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHO5X0q6zavyMuO75xgQIt8trhEUA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041f3-9871-7ea0-bc14-59da58d35efd-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '2026 stock price target analysts highe

In [ ]:
from IPython.display import display, Markdown

display(Markdown(response['messages'][-1].content))

아래는 **2026년 기준 미국주식 중 상승 가능성이 높다고 평가되는 종목들**을, 최근 애널리스트 리포트/컨센서스 목표주가 기준으로 **직관적으로 정리한 표**입니다.  
(※ 실제 투자판단 전에는 최신 실적, 밸류에이션, 리스크를 꼭 확인하세요.)

| 분석기관명 | 목표주가범위 (최저 ~ 최대) | 전망근거 키워드 | 신뢰도 지수(1 ~ 10) |
|---|---:|---|---:|
| MarketBeat/애널리스트 컨센서스 (NVIDIA) | \$218 ~ \$500 | AI 수요, 데이터센터, 고성장, 컨센서스 상향, HBM 수혜 | 9 |
| MarketBeat/애널리스트 컨센서스 (Micron) | \$900 ~ \$2,000 | HBM4, 메모리 슈퍼사이클, AI 반도체, 공급제약, 장기계약 | 8 |
| MarketBeat/애널리스트 컨센서스 (Vistra) | \$187 ~ \$230.31 | AI 전력수요, 유틸리티 수혜, 안정적 현금흐름, 높은 업사이드 | 8 |
| Forbes 인베스터 허브 (Genpact) | 컨센서스 기준 약 +36% 업사이드 | AI 전환, 수익성 개선, EPS 성장, 기술서비스, 밸류에이션 매력 | 7 |
| Forbes 인베스터 허브 (Ryanair) | 컨센서스 기준 약 +25.8% 업사이드 | 운임 회복, 비용 통제, 항공수요, 현금흐름, 마진 방어 | 7 |

### 한줄 결론
- **가장 강한 상승 모멘텀 후보**: **NVIDIA, Micron, Vistra**
- **상승 여력은 있으나 방어적/중간형**: **Genpact, Ryanair**
- 2026년 테마 기준으로는 **AI 반도체(NVIDIA, Micron)**와 **AI 전력 인프라(Vistra)**가 가장 유력합니다.

원하시면 다음 단계로  
**“2096년”이 아니라 “2026년” 기준으로 업종별 1위 종목만 추려서, 매수/관망/주의 등급까지 포함한 표**로 다시 정리해드릴게요.

In [ ]:
a = 10
print(eval("5 + 3 + a"))
exec("b = 10")
print(b)

18
10


In [ ]:
from langchain_core.tools import tool

@tool
def simple_calculator(query: str) -> str:
    """
    산술연산을 위한 간단한 계산기
    Args:
        query: 계산식
    Return:
        계산식 결과값

    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    - simple_calculator("4 ** 2 / 8") -> "계산 결과: 2"
    """
    try:
        result = eval(query)
        return f'계산 결과: {query}'
    except Exception as e:
        return f'계산 오류: {str(e)}'

llm = init_chat_model('gpt-5.4-mini')
tools = [simple_calculator]

agent = create_agent(llm, tools, system_prompt='주어진 도구 활용해 답변 생성')

response = agent.invoke({'messages': '48÷2(9+3) 계산'})

pprint(response)
print('='*30)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='48÷2(9+3) 계산', additional_kwargs={}, response_metadata={}, id='e7ee119d-34cf-4679-a73b-f0abfa1f895b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 216, 'total_tokens': 244, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHOZYVVINTnmVHzkwpoANLrHkJtTs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0420f-fc79-77b3-afb4-192c33502afa-0', tool_calls=[{'name': 'simple_calculator', 'args': {'query': '48 / 2 * (9 + 3)'}, 'id': 'call_qi6JFrnASd0rLrxBVFs

In [ ]:
response = agent.invoke({'messages': '전장에서 계란볶음밥 먹다 죽은 사람 누구냐'})

pprint(response)
print('='*30)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='전장에서 계란볶음밥 먹다 죽은 사람 누구냐', additional_kwargs={}, response_metadata={}, id='8302c72c-b6cb-46fb-bd81-5e41b6789f15'),
              AIMessage(content='확인해보면, “전장에서 계란볶음밥 먹다 죽은 사람”은 특정 역사적 인물로 널리 알려진 표현은 아닙니다. 혹시 아래처럼 많이 알려진 사례를 말씀하신 걸 수 있어요:\n\n- **제갈공명(诸葛亮)**: 전장에서 죽은 인물로는 유명하지만, 계란볶음밥과 직접 연결되진 않습니다.\n- **항우/유방 관련 일화**: 음식 관련 설화가 섞여 전달되는 경우가 있습니다.\n- **인터넷 밈/드립**: “전장에서 계란볶음밥 먹다 죽은 사람”은 실제 역사보다 밈처럼 퍼진 표현일 가능성도 큽니다.\n\n원하시면 제가  \n1) **중국 역사 인물 기준으로** 찾는지,  \n2) **한국 인터넷 밈/유머**인지,  \n3) **드라마/소설/게임 속 캐릭터**인지  \n범위를 좁혀서 더 정확히 찾아드릴게요.\n그 표현만으로는 **딱 특정되는 유명 인물은 없습니다.**  \n아마 **인터넷 밈**이거나, **“전장에서 죽은 사람” + “계란볶음밥”**이 섞인 농담일 가능성이 커요.\n\n혹시 이런 뜻으로 물으신 건가요?\n\n- **전투 중 죽은 역사 인물**\n- **계란볶음밥과 관련된 밈/드립**\n- **어느 작품(드라마, 게임, 소설) 속 인물**\n\n원하시면 제가 문맥 맞춰서 **“그 사람이 누구인지”** 바로 찾아서 좁혀드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 389, 'prompt_tokens': 223, 'total_tokens': 612, 'completion_tokens_details': {'accepted_

In [ ]:
import os
import json
from langchain_core.tools import tool

OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

@tool
def get_current_weather(city_name='Seoul', units='metric'):
    """
    OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**
            - 변환예시:
                - 서울 -> Seoul
                - 충남, 충청남도 -> Chungcheongnam-do
                - 부산 -> Busan
        - units: str 온도단위를 설정하는 문자열
            - metric(기본값: 섭씨, 미터)
            - imperial(화씨, 야드)
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    """
    url = f'https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json()

    weather_info = {}

    if response.status_code == 200:
        weather_description = data['weather'][0]['description']
        temp = data['main']['temp']
        temp_feels_like = data['main']['feels_like']
        humidity = data['main']['humidity']

        weather_info = {
            'city': city_name,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }
    else:
        weather_info = {
            'city': city_name,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }

    return json.dumps(weather_info)

get_current_weather

StructuredTool(name='get_current_weather', description='OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수\n\nArgs:\n    - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**\n        - 변환예시:\n            - 서울 -> Seoul\n            - 충남, 충청남도 -> Chungcheongnam-do\n            - 부산 -> Busan\n    - units: str 온도단위를 설정하는 문자열\n        - metric(기본값: 섭씨, 미터)\n        - imperial(화씨, 야드)\nReturn:\n    - str: json 형식으로 변환된 현재 날씨 정보', args_schema=<class 'langchain_core.utils.pydantic.get_current_weather'>, func=<function get_current_weather at 0x00000184418345E0>)

In [ ]:
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import requests
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [get_current_weather]

agent = create_agent(llm, tools, system_prompt='주어진 도구 활용해 답변 생성')

response = agent.invoke({'messages': '강원도 사는데 뭐 입어야 하냐'})

pprint(response)
print('='*30)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='강원도 사는데 뭐 입어야 하냐', additional_kwargs={}, response_metadata={}, id='4e50e2e1-5403-4d84-b649-9b5edcb1154b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 297, 'total_tokens': 322, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHOuruc9wQqhHEPHbozEU67TOtUlF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04224-27ad-7a60-a1b4-6c86f536da4d-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city_name': 'Gangwon-do', 'units': 'metric'}, 'id': 'c

In [ ]:
from datetime import datetime
from pytz import timezone

@tool
def get_current_datetime(format: str='%Y-%m-%d %H:%M:%S') -> str:
    """
    한국기준 현재시각정보를 반환하는 함수
    Args:
        format: 날짜/시각 형식 지정
    Return:
        현재시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """
    kst = timezone('Asia/Seoul')
    return datetime.now(kst).strftime(format)

get_current_datetime

StructuredTool(name='get_current_datetime', description='한국기준 현재시각정보를 반환하는 함수\nArgs:\n    format: 날짜/시각 형식 지정\nReturn:\n    현재시각 문자열\n\nget_current_datetime() -> "2026-01-15 12:18:32"', args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'>, func=<function get_current_datetime at 0x000001844189B4C0>)

In [ ]:
@tool
def calculate_age(today_date: str, birth_date: str) -> int:
    """
    오늘날짜, 생년월일을 입력받아 나이를 계산하는 도구
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """

    try:
        today = datetime.strptime(today_date, '%Y-%m-%d')
        birthday = datetime.strptime(birth_date, '%Y-%m-%d')

        age = today.year - birthday.year
        if(today.month, today.day) < (birthday.month, birthday.day):
            age -= 1
        return age
    except ValueError:
        return '올바르지 않은 날짜 형식. yyyy-mm-dd'

calculate_age.invoke({'today_date': '2026-08-27', 'birth_date': '2001-12-05'})

24

In [ ]:
llm = init_chat_model('gpt-5.4-mini')
tools = load_tools(['wikipedia']) + [get_current_weather, calculate_age]

agent = create_agent(llm, tools, system_prompt='주어진 도구 활용해 답변 생성')

response = agent.invoke({'messages': '트럼프 현재 나이'}, config={'recursion_limit': 5})

pprint(response)
print('='*30)
pprint(response['messages'][-1].content)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_tavily import TavilySearch

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

agent = create_agent(llm, tools, checkpointer=InMemorySaver())

response = agent.invoke(
    input = {'messages': [('human'), ('안녕. 넌 누구냐')]},
    config={'configurable': {'thread_id': 100}}
)

print(response['messages'][-1].content)

안녕! 나는 OpenAI가 만든 AI 언어 모델이야.  
질문에 답하고, 글을 쓰고, 아이디어를 정리하고, 번역이나 설명도 도와줄 수 있어.

원하면 편하게 한국어로 계속 이야기해도 돼.


In [ ]:
response = agent.invoke(
    input = {'messages': [('human'), ('이전 대답 그대로 출력')]},
    config={'configurable': {'thread_id': 100}}
)

print(response['messages'][-1].content)

안녕! 나는 OpenAI가 만든 AI 언어 모델이야.  
질문에 답하고, 글을 쓰고, 아이디어를 정리하고, 번역이나 설명도 도와줄 수 있어.

원하면 편하게 한국어로 계속 이야기해도 돼.


In [ ]:
response = agent.invoke(
    input = {'messages': [('human'), ('내 이름은 아무것도 아니다. 기억해')]},
    config={'configurable': {'thread_id': 200}}
)

print(response['messages'][-1].content)

알겠습니다. 이 대화에서는 당신을 **“아무것도 아니다”**라고 부르겠습니다.


In [ ]:
response = agent.invoke(
    input = {'messages': [('human'), ('내 이름?')]},
    config={'configurable': {'thread_id': 200}}
)

print(response['messages'][-1].content)

당신의 이름은 **아무것도 아니다**입니다.


In [ ]:
!pip install -Uqqq langgraph-checkpoint-sqlite

In [9]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()

    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(

        input={'messages': [('human', 'Langchain에 대해 설명')]},
        config={'configurable': {'thread_id': '100'}}
    )

    pprint(response['messages'][-1].content)
    print("=" * 50)

    response = agent.invoke(

        input={'messages': [('human', 'Langgraph에 대해 설명')]},
        config={'configurable': {'thread_id': '100'}}
    )

    pprint(response['messages'][-1].content)

('LangChain은 **대규모 언어 모델(LLM)을 활용한 애플리케이션을 만들기 위한 프레임워크**입니다.  \n'
 '쉽게 말해, ChatGPT 같은 모델을 **그냥 질문-답변용으로 쓰는 것**에서 끝내지 않고, **문서 검색, 외부 API 호출, '
 '데이터베이스 조회, 대화 기억, 여러 단계 작업 처리**까지 연결해 주는 도구입니다.\n'
 '\n'
 '## 한 문장 요약\n'
 '**LLM을 실제 서비스에 쓸 수 있게 “연결하고 조립”해 주는 프레임워크**입니다.\n'
 '\n'
 '---\n'
 '\n'
 '## 왜 필요한가?\n'
 'LLM은 강력하지만 단독으로는 한계가 있습니다.\n'
 '\n'
 '- 최신 정보를 항상 아는 것은 아님\n'
 '- 사내 문서나 개인 데이터에 직접 접근 못함\n'
 '- 복잡한 업무를 단계적으로 처리하기 어려움\n'
 '- 외부 시스템과 자동으로 상호작용하지 못함\n'
 '\n'
 'LangChain은 이런 한계를 보완해서  \n'
 '**“모델 + 데이터 + 도구 + 흐름”**을 묶어 하나의 앱처럼 만들 수 있게 합니다.\n'
 '\n'
 '---\n'
 '\n'
 '## LangChain으로 할 수 있는 것\n'
 '### 1. 프롬프트 템플릿 관리\n'
 '자주 쓰는 질문 형식을 재사용할 수 있습니다.\n'
 '\n'
 '### 2. 체인(Chain) 구성\n'
 '여러 작업을 순서대로 연결합니다.  \n'
 '예: 문서 검색 → 핵심 요약 → 최종 답변\n'
 '\n'
 '### 3. RAG 구현\n'
 '외부 문서, PDF, DB에서 관련 내용을 찾아 답변에 반영합니다.\n'
 '\n'
 '### 4. 에이전트(Agent)\n'
 'LLM이 상황에 따라 검색, 계산, API 호출 같은 도구를 선택해 사용합니다.\n'
 '\n'
 '### 5. 메모리(Memory)\n'
 '대화 이력을 유지해서 이어지는 대화를 자연스럽게 처리합니다.\n'
 '\n'
 '### 6. 툴 연결\n'
 '검색엔진, 벡터DB,

In [10]:
response = agent.invoke(

    input={'messages': [('human', 'Langgraph에 대해 설명')]},
    config={'configurable': {'thread_id': '100'}}
)

pprint(response['messages'][-1].content)

ProgrammingError: Cannot operate on a closed database.

In [11]:
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer_tuple = checkpointer.get_tuple({
        'configurable': {
            'thread_id': '100'
        }
    })

    checkpointer_data = checkpointer_tuple.checkpoint
    messages = checkpointer_data['channel_values']['messages']

    for i, message in enumerate(messages, 1):
        msg_type = getattr(message, 'type', message.__class__.__name__)
        print(f'{i}: [{msg_type}] {message.content}')
        print()

1: [human] Langchain에 대해 설명

2: [ai] LangChain은 **대규모 언어 모델(LLM)을 쉽게 연결하고 활용하기 위한 개발 프레임워크**입니다.  
즉, ChatGPT 같은 모델을 그냥 호출하는 데서 끝나는 게 아니라, **외부 데이터, 도구, 메모리, 검색 시스템, 에이전트 로직** 등을 엮어서 실제 서비스처럼 만들 수 있게 도와줍니다.

## 한 줄 요약
**LLM을 “대화형 모델”에서 “실제 업무를 수행하는 애플리케이션”으로 바꾸기 위한 도구 모음**입니다.

---

## LangChain으로 할 수 있는 것
1. **프롬프트 관리**
   - 입력 형식을 체계적으로 만들고 재사용할 수 있음

2. **체인(Chain) 구성**
   - 예: 문서 요약 → 질문 생성 → 답변 생성 같은 여러 단계를 연결

3. **RAG(검색증강생성)**
   - 회사 문서, PDF, DB 등 외부 지식을 검색해서 답변에 반영

4. **에이전트(Agent)**
   - 모델이 필요에 따라 검색, 계산, API 호출 같은 도구를 선택해서 사용

5. **메모리(Memory)**
   - 대화 기록이나 상태를 유지해서 연속적인 대화 구현

6. **도구 연동**
   - 웹 검색, 데이터베이스, 사내 API, 계산기 등과 연결 가능

---

## 왜 쓰는가?
LLM만 단독으로 쓰면:
- 최신 정보에 약할 수 있고
- 회사 내부 문서를 모르며
- 복잡한 작업을 단계적으로 처리하기 어렵습니다.

LangChain은 이런 한계를 줄이고:
- **검색**
- **문서 기반 QA**
- **자동화**
- **업무 보조 챗봇**
같은 걸 만들기 쉽게 해줍니다.

---

## 간단한 예시
예를 들어 “사내 규정에 따르면 휴가 몇 일인가요?”라는 질문에 답하려면:

1. 사용자가 질문
2. LangChain이 사내 규정 문서에서 관련 부분 검색
3. 검색된 내용을 LLM에 전달
4. LLM이 문서를 근거로 답변 생성

이런 흐름을 쉽게 구성할 수 있습니

## Middleware
https://docs.langchain.com/oss/python/langchain/middleware

미들웨어를 통해 에이전트의 추론 과정 중간에 개입하여 내부 동작을 커스터마이징할 수 있다.
Agent를 세부적으로 커스터마이징하기 위한 대부분의 작업을 미들웨어로 할 수 있다.

- 대화 기록 요약
- 동작 중 사용자 입력 대기
- 특정 모델 또는 tool에 대한 호출 제약
- fallback
- PII(개인식별정보) 처리 등

### SummarizationMiddleware

In [13]:
from langchain.agents.middleware import SummarizationMiddleware

model = init_chat_model('gpt-5.4-mini')
summary_model = init_chat_model('gpt-5.4-mini')

middleware_summarize = SummarizationMiddleware(
    model=summary_model,
    trigger=('tokens', 1000),
    keep=('messages', 1),
    summary_prompt='다음 대화 내용 적절히 요약\n{messages}'
)

agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[middleware_summarize]
)

In [14]:
response = agent.invoke(
    input = {'messages': [('human', '뮤지컬 Wicked 내용을 Elphaba 입장에서 서술. Elphaba역으로 연극에 출연해야되서 준비 중. 중요한 포인트들 설명')]},
    config={'configurable': {'thread_id': '5'}}
)

pprint(response)
print('=' * 50)
pprint(response['messages'][-1].content)
print('=' * 50)

response = agent.invoke(
    input = {'messages': [('human', 'Elphaba가 Glinda를 어떤 감정으로 보냐')]},
    config={'configurable': {'thread_id': '5'}}
)

pprint(response)
print('=' * 50)
pprint(response['messages'][-1].content)
print('=' * 50)

response = agent.invoke(
    input = {'messages': [('human', '메소드 연기 필요 부분 파악')]},
    config={'configurable': {'thread_id': '5'}}
)

pprint(response)
print('=' * 50)
pprint(response['messages'][-1].content)
print('=' * 50)

{'messages': [HumanMessage(content='뮤지컬 Wicked 내용을 Elphaba 입장에서 서술. Elphaba역으로 연극에 출연해야되서 준비 중. 중요한 포인트들 설명', additional_kwargs={}, response_metadata={}, id='e369fb28-df6f-4bf2-be46-d703dd3159bb'),
              AIMessage(content='물론입니다. **뮤지컬 _Wicked_를 Elphaba의 입장에서 이해하고 연기 준비할 때 중요한 포인트**를 중심으로 정리해드릴게요.  \n핵심은 **“Elphaba는 악인이 아니라, 오해받고 끝까지 자기 신념을 지키려는 사람”**이라는 점입니다.\n\n---\n\n## 1) Elphaba의 이야기: 한 문장으로\n**초록 피부 때문에 평생 소외받았지만, 정의감과 연민이 강한 Elphaba가 권력의 부조리를 깨닫고 결국 ‘서쪽의 사악한 마녀’로 오해받는 여정**입니다.\n\n---\n\n## 2) Elphaba 시점으로 보는 줄거리\n\n### 1막\nElphaba는 태어날 때부터 초록 피부를 가지고 태어나며, 가족에게도 사랑과 거리를 동시에 경험합니다.  \n어릴 때부터 “다르다”는 이유로 상처를 받고, 사람들의 시선에 익숙해집니다. 하지만 그 안에서 오히려 **정의감, 예민함, 강한 자존심**이 자랍니다.\n\nShiz에서 Glinda와 만나는데, 둘은 처음엔 완전히 반대 성격으로 충돌합니다.  \nGlinda는 인기도와 외향성, 사회적 적응에 강하고, Elphaba는 직설적이고 거칠며, 겉으로는 틱틱대지만 사실 누구보다 진심입니다.\n\nElphaba는 마법적 재능을 인정받고, 결국 Wizard를 만나기 위해 Emerald City로 향합니다.  \n그 과정에서 처음에는 “세상을 바꿀 수 있다”는 희망을 품지만, Wizard와 Madam Morrible의 시스템이 사실은 **조작과 선전, 억압**에 기반해 있다는 걸 알게 됩니다.\n\n가장 큰 전

### PIIMiddleware
**PII (Personally Identifiable Information) 개인식별정보 처리**

https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection

In [15]:
from langchain.agents.middleware import PIIMiddleware

middleware_email = PIIMiddleware(
    pii_type='email',
    detector=r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    strategy='redact',
    apply_to_input=True
)

middleware_credit_card = PIIMiddleware(
    pii_type='credit_card',
    detector=r'(?:\d{4}[-\s]?){3}\d{4}|\d{4}[-\s]?\d{6}[-\s]?\d{5}',
    strategy='redact',
    apply_to_input=True
)

middleware_api_key = PIIMiddleware(
    pii_type='api_key',
    detector=r"sk-[a-zA-Z0-9-_]{161}",
    strategy='redact',
    apply_to_input=True
)

agent = create_agent(
    model=init_chat_model('gpt-5.4-mini'),
    tools=[],
    middleware=[middleware_api_key, middleware_credit_card, middleware_email]
)

In [16]:
response = agent.invoke({
    'messages': [('human', 'AWS 사용하는데 요금 잘못 청구된 것 같다. 금액 재조정 요청하는 메일 영어로 작성. 내 이메일은 capybara@gmail.com이다')]
})

In [17]:
pprint(response)

{'messages': [HumanMessage(content='AWS 사용하는데 요금 잘못 청구된 것 같다. 금액 재조정 요청하는 메일 영어로 작성. 내 이메일은 [REDACTED_EMAIL]이다', additional_kwargs={}, response_metadata={}, id='d8957005-f6b2-44d6-9b68-34a1e87f50df'),
              AIMessage(content='물론입니다. 아래처럼 정중하고 간단하게 보내시면 됩니다.\n\n**Subject:** Request for Billing Review and Adjustment\n\nHello AWS Support Team,\n\nI believe I may have been incorrectly charged for my AWS account. I would appreciate it if you could review my billing details and adjust the amount if necessary.\n\nMy email address is: **[REDACTED_EMAIL]**\n\nPlease let me know if you need any additional information from me to investigate this issue.\n\nThank you for your assistance.\n\nBest regards,  \n[Your Name]\n\n원하시면 제가 이걸 **더 강하게 항의하는 톤** 또는 **아주 공손한 톤**으로도 바꿔드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 43, 'total_tokens': 190, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens

In [20]:
response = agent.invoke({
    'messages': [('human', '최근 쿠팡에서 구매한 100만원 짜리 피규어 환불처리 요청해. 카드번호는 1234-1243-1234-2314이다.')]
})

pprint(response)

{'messages': [HumanMessage(content='최근 쿠팡에서 구매한 100만원 짜리 피규어 환불처리 요청해. 카드번호는 [REDACTED_CREDIT_CARD]이다.', additional_kwargs={}, response_metadata={}, id='e82858b4-6573-4c4c-ab0f-ba25fa387708'),
              AIMessage(content='직접 환불 요청을 대신 처리하진 못하지만, 쿠팡에 바로 보낼 수 있는 **환불 요청 문구**를 만들어드릴게요.\n\n다만 **카드번호 같은 민감한 결제정보는 채팅에 남기지 않는 것이 안전**합니다. 쿠팡 앱/웹의 주문내역에서 해당 주문을 선택해 환불 요청을 진행하세요.\n\n### 쿠팡 고객센터/판매자에게 보낼 문구 예시\n> 최근 쿠팡에서 구매한 피규어에 대해 환불 요청드립니다.  \n> 주문 상품에 대한 환불을 진행하고 싶습니다.  \n> 주문번호 확인 후 처리 부탁드립니다.  \n> 필요 시 반품 절차도 안내해 주세요.\n\n### 쿠팡 앱에서 직접 환불 요청하는 방법\n1. **쿠팡 앱** 실행\n2. **마이쿠팡** → **주문목록**\n3. 해당 피규어 주문 선택\n4. **교환/반품 신청** 또는 **환불 요청** 선택\n5. 사유 선택 후 접수\n\n원하시면 제가  \n- **환불 사유별 문구**(단순변심/불량/오배송/파손)  \n- **고객센터 문의용 더 정중한 버전**  \n으로 바로 작성해드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 296, 'prompt_tokens': 43, 'total_tokens': 339, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejec

## Streaming
*openai모델은 조직인증된 사용자에 한해서 stream기능을 사용할수 있다.*

In [25]:
model = init_chat_model('gpt-5.4-mini')
agent = create_agent(model)
stream = agent.stream(
    input={'messages': [('human', '역사상 가장 위대한 한국인 3명과 점수 제시')]},
    stream_mode='messages'
)

for chunk, metadata in stream:
    print(chunk.content, end='', flush=True)

“역사상 가장 위대한 한국인”은 **기준에 따라 달라지는 주관적 평가**입니다.  
아래는 **영향력, 역사적 파급력, 현재까지의 상징성**을 기준으로 뽑은 제 개인적 순위입니다.

### 1. 세종대왕 — 100점
- 한글 창제라는 인류사적 업적
- 백성의 문자 접근성을 획기적으로 높임
- 한국 문화 정체성의 핵심을 만든 인물

### 2. 이순신 — 98점
- 임진왜란에서 국가를 지켜낸 상징적 인물
- 뛰어난 전략가이자 리더
- 한국인이 가장 존경하는 역사적 인물 중 하나

### 3. 김구 — 95점
- 독립운동의 상징적 지도자
- 민족 자주성과 독립정신의 대표
- 오늘날까지도 정치·사회적 가치의 기준점으로 언급됨

원하시면 제가 이어서  
- **“역사상 가장 위대한 한국인 TOP 10”**  
- **“분야별(과학/군사/정치/문화) 최고 한국인”**  
형태로도 정리해드릴 수 있습니다.